# Phase 3 — trade imbalance

See `STRATEGY.md` for the full 6-phase plan. Phase 2 found that relative strength (a
cross-sectional z-score of trailing return) does almost all the work of making the Phase 1
reversal signal selective — VWAP distance and volume surprise, both derived from the same
OHLCV data, added very little beyond it. This phase adds a genuinely different source of
information: **trade imbalance** — whether recent trading volume was predominantly
buyer-initiated or seller-initiated — which price/volume bars alone cannot express.

**Classification method, deliberately the simpler one**: each trade tick is classified by the
**tick rule** (an uptick from the previous trade is a buy, a downtick is a sell, an unchanged
price inherits the previous classification) rather than Lee-Ready (which compares each trade to
the prevailing bid/ask and needs matching quote data). See `download_trade_ticks.py`'s docstring
for why — it roughly halves the data that needed to be pulled, and the tick rule is a
long-established classification method in its own right, just less precise.

**Trade imbalance (TOI)** per minute: `(buy_volume - sell_volume) / (buy_volume + sell_volume)`,
aggregated from tick-level classified volume into the same 1-minute bins the Phase 1/2 bars use.

**Scope, much smaller than Phase 1/2**: the tick cache covers only ~1 trading day (vs. Phase
1-2's 1 month of minute bars) — see `download_trade_ticks.py`'s docstring for the feasibility
math (a sustained pull measured ~1.8s/chunk, so even 5 days across 131 names of mixed liquidity
would have taken hours). A single day of tick data still has enormous observation density
(liquid names alone produce hundreds of thousands of trades/day), but this is a much smaller,
much more recent, single-regime sample than Phase 1/2 — treat findings here as more provisional
than those, not as directly comparable in magnitude.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MINUTE_DATA_DIR = Path('data') / 'minute'
TICK_DATA_DIR = Path('data') / 'ticks'

K, H = 1, 1   # same cell Phase 1/2 found strongest
N_BUCKETS = 3
COST_BPS_GRID = [0, 1, 2, 5, 10]
TOP_FRACTION = 0.1

## Classify ticks and build per-minute trade imbalance

In [ ]:
def load_and_classify_ticks(ticker):
    frame = pd.read_csv(TICK_DATA_DIR / f'{ticker}.csv', parse_dates=['time'])
    frame['time'] = pd.to_datetime(frame['time'], utc=True)
    frame = frame.sort_values('time').reset_index(drop=True)
    diff = frame['price'].diff()
    direction = np.sign(diff).replace(0, np.nan).ffill()
    direction = direction.fillna(1.0)  # the very first trade has no prior tick to compare to; arbitrary, negligible given volume
    frame['direction'] = direction
    frame['minute'] = frame['time'].dt.floor('min')
    return frame

tick_tickers = sorted(p.stem for p in TICK_DATA_DIR.glob('*.csv'))
print(f'{len(tick_tickers)} tickers with tick data cached')

toi_frames = []
for ticker in tick_tickers:
    ticks = load_and_classify_ticks(ticker)
    buy_vol = ticks.loc[ticks['direction'] > 0].groupby('minute')['size'].sum()
    sell_vol = ticks.loc[ticks['direction'] < 0].groupby('minute')['size'].sum()
    vol = pd.DataFrame({'buy_vol': buy_vol, 'sell_vol': sell_vol}).fillna(0.0)
    vol['toi'] = (vol['buy_vol'] - vol['sell_vol']) / (vol['buy_vol'] + vol['sell_vol'])
    vol['ticker'] = ticker
    toi_frames.append(vol)

toi_panel = pd.concat(toi_frames)
toi_panel.index.name = 'minute'
print(f'{len(toi_panel):,} (ticker, minute) trade-imbalance observations, '
      f'{toi_panel.index.min()} to {toi_panel.index.max()}')

## Merge with the Phase 1/2 minute-bar features

Reuses the exact same reversal/VWAP/volume-surprise/relative-strength construction as Phase 2, restricted to tickers that also have tick data cached, then joins in `toi` by (ticker, minute).

In [ ]:
def load_bar(ticker):
    frame = pd.read_csv(MINUTE_DATA_DIR / f'{ticker}.csv', parse_dates=['date'])
    frame['date'] = pd.to_datetime(frame['date'], utc=True)
    frame = frame.sort_values('date').set_index('date')
    frame['session_date'] = frame.index.tz_convert('America/New_York').date
    frame['minute_of_session'] = frame.groupby('session_date').cumcount()
    return frame

bar_frames = []
for ticker in tick_tickers:
    bar_path = MINUTE_DATA_DIR / f'{ticker}.csv'
    if not bar_path.exists():
        continue
    df = load_bar(ticker)
    g = df.groupby('session_date')

    trail = g['close'].transform(lambda s: s / s.shift(K) - 1)
    fwd = g['close'].transform(lambda s: s.shift(-H) / s - 1)
    reversal_score = -trail

    pv = df['average'] * df['volume']
    cum_pv = pv.groupby(df['session_date']).cumsum()
    cum_vol = df['volume'].groupby(df['session_date']).cumsum()
    vwap_dist = df['close'] / (cum_pv / cum_vol) - 1

    baseline_vol = df.groupby('minute_of_session')['volume'].transform('mean')
    volume_surprise = df['volume'] / baseline_vol - 1

    bar_frames.append(pd.DataFrame({
        'ticker': ticker, 'trail': trail, 'fwd': fwd, 'reversal_score': reversal_score,
        'vwap_dist': vwap_dist, 'volume_surprise': volume_surprise,
    }, index=df.index))

bars = pd.concat(bar_frames).dropna(subset=['reversal_score', 'fwd', 'vwap_dist', 'volume_surprise'])
cs_mean = bars.groupby(bars.index)['trail'].transform('mean')
cs_std = bars.groupby(bars.index)['trail'].transform('std')
bars['relative_strength'] = (bars['trail'] - cs_mean) / cs_std
bars = bars.dropna(subset=['relative_strength'])

panel = bars.reset_index().merge(
    toi_panel.reset_index().rename(columns={'minute': 'date'}),
    on=['ticker', 'date'], how='inner',
).set_index('date')

print(f'{len(panel):,} (ticker, minute) observations with both bar features and trade imbalance '
      f'({panel["ticker"].nunique()} tickers, {panel.index.min()} to {panel.index.max()})')

## Does trade imbalance predict the reversal, beyond what Phase 2 already found?

In [ ]:
baseline_ic = panel['reversal_score'].corr(panel['fwd'])
print(f'unconditional IC on this (smaller, 1-day) sample: {baseline_ic:+.4f}')

FEATURES = ['vwap_dist', 'volume_surprise', 'relative_strength', 'toi']
bucket_rows = []
for feat in FEATURES:
    try:
        buckets = pd.qcut(panel[feat].abs(), N_BUCKETS, labels=['low', 'mid', 'high'], duplicates='drop')
    except ValueError:
        continue
    for label, group in panel.groupby(buckets, observed=True):
        bucket_rows.append({'feature': feat, 'bucket': label, 'n': len(group),
                             'ic': group['reversal_score'].corr(group['fwd'])})
bucket_table = pd.DataFrame(bucket_rows)
display(bucket_table.pivot(index='feature', columns='bucket', values='ic')[['low', 'mid', 'high']].round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
pivot = bucket_table.pivot(index='feature', columns='bucket', values='ic')[['low', 'mid', 'high']]
pivot.plot(kind='bar', ax=ax, color=['#c0392b', '#888888', '#176b87'])
ax.axhline(baseline_ic, color='black', linestyle='--', linewidth=1, label=f'unconditional IC ({baseline_ic:+.3f})')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('IC within bucket')
ax.set_title('Reversal signal IC, conditioned on each feature\'s magnitude tercile (incl. trade imbalance)')
ax.legend()
plt.tight_layout()
plt.show()

## Combined filter: does adding trade imbalance to Phase 2's filter help?

Compares three nested filters: unconditional, Phase 2's filter (relative_strength + vwap_dist + volume_surprise all high), and that same filter plus trade imbalance also required high.

In [ ]:
phase2_features = ['vwap_dist', 'volume_surprise', 'relative_strength']
all_features = phase2_features + ['toi']

def combined_filter(data, features):
    mask = pd.Series(True, index=data.index)
    for feat in features:
        mask &= data[feat].abs() >= data[feat].abs().quantile(2 / 3)
    return data[mask]

phase2_filtered = combined_filter(panel, phase2_features)
phase3_filtered = combined_filter(panel, all_features)

for name, data in [('unconditional', panel), ('Phase 2 filter (no TOI)', phase2_filtered), ('Phase 3 filter (+ TOI)', phase3_filtered)]:
    ic = data['reversal_score'].corr(data['fwd']) if len(data) > 30 else float('nan')
    print(f'{name}: n={len(data):,} ({len(data)/len(panel):.1%} of sample), IC={ic:+.4f}')

## Backtest comparison

In [ ]:
def decile_backtest(data, min_names=5):
    results = []
    for minute, group in data.groupby(data.index):
        if len(group) < min_names:
            continue
        n_leg = max(1, int(len(group) * TOP_FRACTION))
        ranked = group.sort_values('reversal_score')
        bottom_leg, top_leg = ranked.iloc[:n_leg], ranked.iloc[-n_leg:]
        spread_return = top_leg['fwd'].mean() - bottom_leg['fwd'].mean()
        results.append({'minute': minute, 'spread_return': spread_return, 'n_names': len(group)})
    if not results:
        return None
    return pd.DataFrame(results).set_index('minute').sort_index()

for name, data in [('unconditional', panel), ('Phase 2 filter (no TOI)', phase2_filtered), ('Phase 3 filter (+ TOI)', phase3_filtered)]:
    bt = decile_backtest(data)
    if bt is None or len(bt) < 10:
        print(f'{name}: not enough minutes with >= 5 names to backtest on this 1-day sample')
        continue
    mean_spread = bt['spread_return'].mean()
    print(f'{name}: {len(bt):,} decisions, avg {bt["n_names"].mean():.1f} names/cross-section, '
          f'gross mean spread = {mean_spread*10_000:+.2f} bps')
    for cost in COST_BPS_GRID:
        net = mean_spread - 2 * (cost / 10_000.0)
        print(f'    at {cost}bps cost: net = {net*10_000:+.2f} bps')

## Caveats

_placeholder — filled in after running this notebook end-to-end_